In [1]:
import sys
sys.path.append('/home/nicolo_b/Desktop/PhD/RELIABLE_NAS/NOTEBOOK/FAULT_INJECTOR/VITTFI')

import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

from FaultGenerators.WeightFault import WeightFault
from FaultGenerators.WeightFaultInjector import WeightFaultInjector
from BERCampaign import BERCampaign

In [2]:
class ToyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool  = nn.AdaptiveAvgPool2d((1, 1))
        self.fc    = nn.Linear(16, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).flatten(1)
        return self.fc(x)

In [3]:
torch.manual_seed(42)
images = torch.randn(200, 3, 8, 8)
labels = torch.randint(0, 10, (200,))
dataset = TensorDataset(images, labels)
loader  = DataLoader(dataset, batch_size=50)

In [4]:
device   = torch.device('cpu')
network  = ToyNet().to(device)
injector = WeightFaultInjector(network)

campaign = BERCampaign(
    network          = network,
    loader           = loader,
    injector         = injector,
    device           = device,
    injection_levels = [1, 10, 50, 100],
    sampling_mode    = 'constant',
    pilot_trials     = 10,
    max_trials       = 50,
    precision_e      = 0.01,
    confidence_t     = 2.576,
    seed             = 42,
)

In [5]:
print(f"layer_names   : {campaign.layer_names}")
print(f"layer_shapes  : {campaign.layer_shapes}")
print(f"layer_offsets : {campaign.layer_offsets}")
print(f"total_weights : {campaign.total_weights}")
print(f"total_bits    : {campaign.total_bits}")

layer_names   : ['conv1', 'conv2']
layer_shapes  : [(8, 3, 3, 3), (16, 8, 3, 3)]
layer_offsets : [  0 216]
total_weights : 1368
total_bits    : 43776


In [6]:
results = campaign.run()

[BERCampaign] level=1 | trials=10 | mean=0.0650 | std=0.0000 | half_width=0.0000
[BERCampaign] level=10 | trials=10 | mean=0.0670 | std=0.0063 | half_width=0.0052
[BERCampaign] level=50 | trials=10 | mean=0.0825 | std=0.0103 | half_width=0.0084
[BERCampaign] level=100 | trials=14 | mean=0.0907 | std=0.0128 | half_width=0.0088


In [7]:
for level, r in results.items():
    print(f"\ninjection_level = {level}")
    print(f"  golden_accuracy      = {r['golden_accuracy']:.4f}")
    print(f"  mean faulty acc      = {r['mean']:.4f}")
    print(f"  std                  = {r['std']:.4f}")
    print(f"  n_trials             = {r['n_trials']}")
    print(f"  n_target             = {r['n_target']}")
    print(f"  effective_half_width = {r['effective_half_width']:.4f}")


injection_level = 1
  golden_accuracy      = 0.0650
  mean faulty acc      = 0.0650
  std                  = 0.0000
  n_trials             = 10
  n_target             = 10
  effective_half_width = 0.0000

injection_level = 10
  golden_accuracy      = 0.0650
  mean faulty acc      = 0.0670
  std                  = 0.0063
  n_trials             = 10
  n_target             = 10
  effective_half_width = 0.0052

injection_level = 50
  golden_accuracy      = 0.0650
  mean faulty acc      = 0.0825
  std                  = 0.0103
  n_trials             = 10
  n_target             = 10
  effective_half_width = 0.0084

injection_level = 100
  golden_accuracy      = 0.0650
  mean faulty acc      = 0.0907
  std                  = 0.0128
  n_trials             = 14
  n_target             = 14
  effective_half_width = 0.0088
